In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Project directory
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Data directories
RAW_DATA = PROJECT_DIR / "data" / "raw"
CSV_DIR = RAW_DATA / "csv"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("KAWASAN Dataset Catalogue")
print("Project directory:", PROJECT_DIR)
print("CSV directory:", CSV_DIR)


KAWASAN Dataset Catalogue
Project directory: c:\Users\Muhd Haziq\KAWASAN
CSV directory: c:\Users\Muhd Haziq\KAWASAN\data\raw\csv


In [2]:

# Dataset Inventory

dataset_files = sorted(CSV_DIR.glob("*.csv"))

inventory = []

for file in dataset_files:
    df = pd.read_csv(file)

    inventory.append({
        "dataset": file.name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isna().sum().sum(),
        "column_names": ", ".join(df.columns)
    })

dataset_inventory = pd.DataFrame(inventory)

print("Total datasets:", len(dataset_inventory))
display(dataset_inventory)


Total datasets: 25


,dataset,rows,columns,missing_values,column_names
0,census_district.csv,960,51,20832,"state, code_state, district, code_district, co..."
1,completion_school_district.csv,2355,5,5,"state, district, stage, date, completion"
2,cpi_2d_state.csv,45024,4,0,"state, date, division, index"
3,enrolment_school_district.csv,12756,6,0,"state, district, stage, sex, date, students"
4,formal_wages_state.csv,918,4,0,"state, date, employees, wages_median"
5,gdp_district_real_supply.csv,10626,6,1238,"series, state, district, date, sector, value"
6,gdp_state_real_supply.csv,2163,5,1,"series, date, state, sector, value"
7,hh_access_amenities.csv,694,6,55,"date, state, district, piped_water, sanitation..."
8,hh_access_internet.csv,343,4,0,"date, state, strata, proportion"
9,hh_income_district.csv,480,5,0,"state, district, date, income_mean, income_median"


In [3]:

coverage = []

for file in sorted(CSV_DIR.glob("*.csv")):
    df = pd.read_csv(file, low_memory=False)

    if "district" in df.columns:
        geographic_level = "District"
    elif "state" in df.columns:
        geographic_level = "State"
    elif "premise_code" in df.columns:
        geographic_level = "Premise-level (requires lookup)"
    else:
        geographic_level = "Other / National"

    if "date" in df.columns:
        dates = pd.to_datetime(df["date"], errors="coerce")
        start_year = dates.min().year if dates.notna().any() else None
        end_year = dates.max().year if dates.notna().any() else None
    else:
        start_year = None
        end_year = None

    coverage.append({
        "dataset": file.name,
        "geographic_level": geographic_level,
        "start_year": start_year,
        "end_year": end_year
    })

coverage_df = pd.DataFrame(coverage)

display(coverage_df)


,dataset,geographic_level,start_year,end_year
0,census_district.csv,District,NaN,NaN
1,completion_school_district.csv,District,2021.0,2025.0
2,cpi_2d_state.csv,State,2010.0,2026.0
3,enrolment_school_district.csv,District,2017.0,2025.0
4,formal_wages_state.csv,State,2022.0,2026.0
5,gdp_district_real_supply.csv,District,2015.0,2020.0
6,gdp_state_real_supply.csv,State,2015.0,2025.0
7,hh_access_amenities.csv,District,2016.0,2024.0
8,hh_access_internet.csv,State,2019.0,2025.0
9,hh_income_district.csv,District,2019.0,2024.0


In [4]:

gdp = pd.read_csv(CSV_DIR / "gdp_district_real_supply.csv")

print("Shape:", gdp.shape)

print("\nColumns:")
print(gdp.columns.tolist())

print("\nMissing values:")
display(gdp.isnull().sum())

print("\nReporting years:")
print(sorted(gdp["date"].astype(str).str[:4].unique()))

print("\nGDP series:")
display(gdp["series"].value_counts())

print("\nEconomic sectors:")
display(gdp["sector"].value_counts())

print("\nSample records:")
display(gdp.head(10))



Shape: (10626, 6)

Columns:
['series', 'state', 'district', 'date', 'sector', 'value']

Missing values:


series         0
state          0
district       0
date           0
sector         0
value       1238
dtype: int64


Reporting years:
['2015', '2016', '2017', '2018', '2019', '2020']

GDP series:


series
abs           5796
growth_yoy    4830
Name: count, dtype: int64


Economic sectors:


sector
p0    1771
p1    1771
p2    1771
p3    1771
p4    1771
p5    1771
Name: count, dtype: int64


Sample records:


,series,state,district,date,sector,value
0,abs,Johor,Batu Pahat,2015-01-01,p0,11605.5017
1,abs,Johor,Batu Pahat,2015-01-01,p1,2177.2570
2,abs,Johor,Batu Pahat,2015-01-01,p2,13.0113
3,abs,Johor,Batu Pahat,2015-01-01,p3,3255.7651
4,abs,Johor,Batu Pahat,2015-01-01,p4,109.0029
5,abs,Johor,Batu Pahat,2015-01-01,p5,5915.2628
6,abs,Johor,Batu Pahat,2016-01-01,p0,12278.8487
7,abs,Johor,Batu Pahat,2016-01-01,p1,2226.4352
8,abs,Johor,Batu Pahat,2016-01-01,p2,15.7649
9,abs,Johor,Batu Pahat,2016-01-01,p3,3460.4801


In [5]:

gdp["year"] = pd.to_datetime(gdp["date"]).dt.year

print("1. Missing values by GDP series")
display(
    gdp.groupby("series")["value"]
    .agg(total="size", missing="count")
    .assign(missing=lambda x: x["total"] - x["missing"])
)

print("\n2. Missing values by year")
display(
    gdp.groupby("year")["value"]
    .agg(total="size", available="count", missing=lambda x: x.isna().sum())
)

print("\n3. Missing values by economic sector")
display(
    gdp.groupby("sector")["value"]
    .agg(total="size", available="count", missing=lambda x: x.isna().sum())
)

print("\n4. Missing values by GDP series and sector")
display(
    gdp.pivot_table(
        index="sector",
        columns="series",
        values="value",
        aggfunc=lambda x: x.isna().sum()
    )
)

print("\n5. District coverage by year")
display(
    gdp.groupby("year")["district"]
    .nunique()
    .rename("unique_districts")
    .to_frame()
)


1. Missing values by GDP series


,total,missing
series,,
abs,5796,507
growth_yoy,4830,731



2. Missing values by year


,total,available,missing
year,,,
2015,966,882,84
2016,1932,1703,229
2017,1932,1706,226
2018,1932,1702,230
2019,1932,1701,231
2020,1932,1694,238



3. Missing values by economic sector


,total,available,missing
sector,,,
p0,1771,1771,0
p1,1771,1745,26
p2,1771,1165,606
p3,1771,1540,231
p4,1771,1406,365
p5,1771,1761,10



4. Missing values by GDP series and sector


series,abs,growth_yoy
sector,,
p0,0,0
p1,6,20
p2,213,393
p3,101,130
p4,187,178
p5,0,10



5. District coverage by year


,unique_districts
year,
2015,160
2016,160
2017,160
2018,160
2019,160
2020,160


In [6]:

# Missing GDP records by district and year

missing_gdp = gdp[gdp["value"].isna()].copy()

print("Total missing records:", len(missing_gdp))

print("\nMissing values by district:")
display(
    missing_gdp.groupby(["state", "district"])
    .size()
    .reset_index(name="missing_records")
    .sort_values("missing_records", ascending=False)
    .head(20)
)

print("\nMissing values by year and sector:")
display(
    missing_gdp.pivot_table(
        index="year",
        columns="sector",
        values="value",
        aggfunc="size",
        fill_value=0
    )
)

# Check duplicate records
duplicate_columns = [
    "series", "state", "district", "date", "sector"
]

print("\nDuplicate records:", gdp.duplicated(subset=duplicate_columns).sum())

print("\nMissing records by series:")
display(missing_gdp["series"].value_counts())


Total missing records: 1238

Missing values by district:


,state,district,missing_records
75,Sarawak,Selangau,27
60,Sarawak,Julau,27
49,Sabah,Tambunan,27
58,Sarawak,Dalat,27
85,Sarawak,Telang Usan,27
84,Sarawak,Tebedu,27
69,Sarawak,Matu,27
71,Sarawak,Pusa,27
61,Sarawak,Kabong,27
41,Sabah,Nabawan,27



Missing values by year and sector:


sector,p1,p2,p3,p4,p5
year,,,,,
2015,1,39,17,27,0
2016,5,119,43,60,2
2017,5,112,43,64,2
2018,5,112,43,68,2
2019,5,111,42,71,2
2020,5,113,43,75,2



Duplicate records: 0

Missing records by series:


series
growth_yoy    731
abs           507
Name: count, dtype: int64

In [7]:

# Inspect unique GDP values and sector patterns

print("GDP series:")
display(gdp[["series"]].drop_duplicates())

print("\nSector codes:")
display(gdp[["sector"]].drop_duplicates())

print("\nGDP values by series:")
display(
    gdp.groupby("series")["value"]
    .describe()
)

print("\nSample GDP records for each sector:")
display(
    gdp.dropna(subset=["value"])
    .groupby(["series", "sector"])
    .head(3)
    .sort_values(["series", "sector"])
)


GDP series:


,series
0,abs
5796,growth_yoy



Sector codes:


,sector
0,p0
1,p1
2,p2
3,p3
4,p4
5,p5



GDP values by series:


,count,mean,std,min,25%,50%,75%,max
series,,,,,,,,
abs,5289.0,2840.303030,12459.185727,0.0,46.3442,326.8607,1364.3216,233793.693
growth_yoy,4099.0,2.229397,15.113500,-100.0,-2.7000,3.3000,6.3000,474.000



Sample GDP records for each sector:


,series,state,district,date,sector,value,year
0,abs,Johor,Batu Pahat,2015-01-01,p0,11605.5017,2015
6,abs,Johor,Batu Pahat,2016-01-01,p0,12278.8487,2016
12,abs,Johor,Batu Pahat,2017-01-01,p0,13115.0995,2017
1,abs,Johor,Batu Pahat,2015-01-01,p1,2177.2570,2015
7,abs,Johor,Batu Pahat,2016-01-01,p1,2226.4352,2016
13,abs,Johor,Batu Pahat,2017-01-01,p1,2333.1613,2017
2,abs,Johor,Batu Pahat,2015-01-01,p2,13.0113,2015
8,abs,Johor,Batu Pahat,2016-01-01,p2,15.7649,2016
14,abs,Johor,Batu Pahat,2017-01-01,p2,18.5291,2017
3,abs,Johor,Batu Pahat,2015-01-01,p3,3255.7651,2015


In [8]:

# Load official DOSM GDP lookup table

lookup_url = "https://storage.dosm.gov.my/gdp/gdp_lookup.csv"

gdp_lookup = pd.read_csv(lookup_url)

# Display production sector definitions
gdp_sector_lookup = gdp_lookup[
    gdp_lookup["method"] == "production"
][["code", "variable_en", "variable_bm"]]

display(gdp_sector_lookup)



KeyError: "['variable_en', 'variable_bm'] not in index"

In [9]:
print("Shape:", gdp_lookup.shape)

print("\nAvailable columns:")
print(gdp_lookup.columns.tolist())

print("\nFirst 10 records:")
display(gdp_lookup.head(10))

print("\nData types:")
display(gdp_lookup.dtypes)

Shape: (174, 4)

Available columns:
['method', 'code', 'desc_en', 'desc_bm']

First 10 records:


,method,code,desc_en,desc_bm
0,production,p0,GDP at purchasers' prices,KDNK pada harga pembeli
1,production,p1,Agriculture,Pertanian
2,production,p1.1,Crops,Tanaman
3,production,p1.1.1,Rubber,Getah
4,production,p1.1.2,Oil palm,Kelapa sawit
5,production,p1.1.3,Paddy,Padi
6,production,p1.1.4,Vegetables,Sayur-sayuran
7,production,p1.1.5,Fruits,Buah-buahan
8,production,p1.1.6,Food crops,Tanaman makanan
9,production,p1.1.7,Others,Lain-lain



Data types:


method     str
code       str
desc_en    str
desc_bm    str
dtype: object

In [10]:
gdp_sector_lookup = gdp_lookup[
    gdp_lookup["method"] == "production"
][["code", "desc_en", "desc_bm"]]

display(gdp_sector_lookup)

,code,desc_en,desc_bm
0,p0,GDP at purchasers' prices,KDNK pada harga pembeli
1,p1,Agriculture,Pertanian
2,p1.1,Crops,Tanaman
3,p1.1.1,Rubber,Getah
4,p1.1.2,Oil palm,Kelapa sawit
...,...,...,...
108,p5.5.07,Health,Kesihatan
109,p5.5.08,"Recreation, culture and religion","Rekreasi, kebudayaan dan keagamaan"
110,p5.5.09,Education,Pendidikan
111,p5.5.10,Social protection,Perlindungan sosial


In [11]:
gdp = gdp.merge(
    gdp_sector_lookup,
    left_on="sector",
    right_on="code",
    how="left"
)

gdp.drop(columns=["code"], inplace=True)

display(
    gdp[[
        "series",
        "state",
        "district",
        "year",
        "sector",
        "desc_en",
        "value"
    ]].head(15)
)

C:\Users\Muhd Haziq\AppData\Local\Temp\ipykernel_18892\3198808666.py:8: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  gdp.drop(columns=["code"], inplace=True)


,series,state,district,year,sector,desc_en,value
0,abs,Johor,Batu Pahat,2015,p0,GDP at purchasers' prices,11605.5017
1,abs,Johor,Batu Pahat,2015,p1,Agriculture,2177.2570
2,abs,Johor,Batu Pahat,2015,p2,Mining and quarrying,13.0113
3,abs,Johor,Batu Pahat,2015,p3,Manufacturing,3255.7651
4,abs,Johor,Batu Pahat,2015,p4,Construction,109.0029
5,abs,Johor,Batu Pahat,2015,p5,Services,5915.2628
6,abs,Johor,Batu Pahat,2016,p0,GDP at purchasers' prices,12278.8487
7,abs,Johor,Batu Pahat,2016,p1,Agriculture,2226.4352
8,abs,Johor,Batu Pahat,2016,p2,Mining and quarrying,15.7649
9,abs,Johor,Batu Pahat,2016,p3,Manufacturing,3460.4801


In [12]:
print("Sector mapping:")
display(
    gdp[["sector", "desc_en"]]
    .drop_duplicates()
    .sort_values("sector")
    .reset_index(drop=True)
)

print("\nUnmapped records:", gdp["desc_en"].isna().sum())

duplicate_columns = [
    "series", "state", "district", "date", "sector"
]

print(
    "Duplicate records:",
    gdp.duplicated(subset=duplicate_columns).sum()
)

Sector mapping:


,sector,desc_en
0,p0,GDP at purchasers' prices
1,p1,Agriculture
2,p2,Mining and quarrying
3,p3,Manufacturing
4,p4,Construction
5,p5,Services



Unmapped records: 0
Duplicate records: 0


In [13]:
print("GDP series summary:")
display(
    gdp.groupby("series")["value"]
    .describe()
    .round(2)
)

print("\nAbsolute GDP by sector:")
display(
    gdp[gdp["series"] == "abs"]
    .groupby("desc_en")["value"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("\nGrowth rate by sector:")
display(
    gdp[gdp["series"] == "growth_yoy"]
    .groupby("desc_en")["value"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

GDP series summary:


,count,mean,std,min,25%,50%,75%,max
series,,,,,,,,
abs,5289.0,2840.30,12459.19,0.0,46.34,326.86,1364.32,233793.69
growth_yoy,4099.0,2.23,15.11,-100.0,-2.70,3.30,6.30,474.00



Absolute GDP by sector:


,count,mean,median,min,max
desc_en,,,,,
Agriculture,960,615.96,392.13,0.00,3574.71
Construction,779,467.70,61.19,0.00,18469.70
GDP at purchasers' prices,966,7826.10,1806.55,53.65,233793.69
Manufacturing,865,2029.33,152.45,0.00,45071.00
Mining and quarrying,753,441.24,10.29,0.00,29008.51
Services,966,4574.60,826.51,0.00,205330.86



Growth rate by sector:


,count,mean,median,min,max
desc_en,,,,,
Agriculture,785,0.66,0.9,-30.7,36.2
Construction,627,2.15,-0.0,-71.0,474.0
GDP at purchasers' prices,805,2.16,3.1,-29.7,38.0
Manufacturing,675,1.50,2.1,-40.6,69.9
Mining and quarrying,412,4.69,7.3,-100.0,34.1
Services,795,3.25,4.4,-29.6,32.0


In [14]:
growth = gdp[gdp["series"] == "growth_yoy"].copy()

print("Highest 15 GDP growth observations:")
display(
    growth.nlargest(15, "value")[
        ["state", "district", "year", "sector", "desc_en", "value"]
    ].reset_index(drop=True)
)

print("\nLowest 15 GDP growth observations:")
display(
    growth.nsmallest(15, "value")[
        ["state", "district", "year", "sector", "desc_en", "value"]
    ].reset_index(drop=True)
)

print("\nGrowth observations outside -50% to 100%:")
display(
    growth[
        (growth["value"] < -50) |
        (growth["value"] > 100)
    ][
        ["state", "district", "year", "sector", "desc_en", "value"]
    ].sort_values("value", ascending=False).reset_index(drop=True)
)

Highest 15 GDP growth observations:


,state,district,year,sector,desc_en,value
0,Melaka,Alor Gajah,2017,p4,Construction,474.0
1,Johor,Kota Tinggi,2016,p4,Construction,140.2
2,Terengganu,Marang,2017,p4,Construction,109.1
3,Sabah,Keningau,2016,p4,Construction,78.8
4,Sarawak,Betong,2017,p4,Construction,74.5
5,Sabah,Penampang,2018,p4,Construction,73.9
6,Sarawak,Lundu,2018,p4,Construction,70.4
7,Kelantan,Kota Bharu,2016,p4,Construction,69.9
8,Sabah,Sipitang,2017,p3,Manufacturing,69.9
9,Johor,Kota Tinggi,2020,p3,Manufacturing,68.6



Lowest 15 GDP growth observations:


,state,district,year,sector,desc_en,value
0,Sarawak,Marudi,2019,p2,Mining and quarrying,-100.0
1,Sarawak,Marudi,2018,p2,Mining and quarrying,-74.7
2,Kelantan,Machang,2016,p4,Construction,-71.0
3,Terengganu,Marang,2020,p4,Construction,-62.6
4,Pahang,Pekan,2019,p4,Construction,-62.2
5,Perak,Manjung,2017,p4,Construction,-62.2
6,Kedah,Baling,2016,p4,Construction,-57.8
7,Sarawak,Simunjan,2017,p4,Construction,-57.8
8,Sabah,Sipitang,2016,p4,Construction,-57.2
9,Johor,Kota Tinggi,2020,p4,Construction,-56.9



Growth observations outside -50% to 100%:


,state,district,year,sector,desc_en,value
0,Melaka,Alor Gajah,2017,p4,Construction,474.0
1,Johor,Kota Tinggi,2016,p4,Construction,140.2
2,Terengganu,Marang,2017,p4,Construction,109.1
3,Pahang,Bera,2019,p4,Construction,-50.1
4,Sabah,Kudat,2019,p4,Construction,-51.0
5,Terengganu,Marang,2016,p4,Construction,-51.1
6,Terengganu,Dungun,2019,p4,Construction,-51.3
7,Sarawak,Pakan,2019,p4,Construction,-51.4
8,Sabah,Tawau,2020,p4,Construction,-52.0
9,Kedah,Kulim,2018,p4,Construction,-52.7


In [15]:
abs_gdp = gdp[gdp["series"] == "abs"][
    ["state", "district", "year", "sector", "value"]
].copy()

extreme_growth = growth[
    (growth["value"] < -50) |
    (growth["value"] > 100)
].copy()

current_gdp = abs_gdp.rename(columns={
    "year": "year_current",
    "value": "gdp_current"
})

previous_gdp = abs_gdp.rename(columns={
    "year": "year_previous",
    "value": "gdp_previous"
})

extreme_check = extreme_growth.merge(
    current_gdp,
    left_on=["state", "district", "year", "sector"],
    right_on=["state", "district", "year_current", "sector"],
    how="left"
)

extreme_check["year_previous"] = extreme_check["year"] - 1

extreme_check = extreme_check.merge(
    previous_gdp,
    on=["state", "district", "year_previous", "sector"],
    how="left"
)

extreme_check["calculated_growth"] = (
    (extreme_check["gdp_current"] /
     extreme_check["gdp_previous"] - 1) * 100
)

display(
    extreme_check[
        [
            "state", "district", "year", "desc_en",
            "gdp_previous", "gdp_current",
            "value", "calculated_growth"
        ]
    ].sort_values("value", ascending=False).round(2)
)

,state,district,year,desc_en,gdp_previous,gdp_current,value,calculated_growth
7,Melaka,Alor Gajah,2017,Construction,116.25,667.27,474.0,474.02
0,Johor,Kota Tinggi,2016,Construction,1665.49,4000.37,140.2,140.19
23,Terengganu,Marang,2017,Construction,29.95,62.62,109.1,109.13
8,Pahang,Bera,2019,Construction,48.90,24.41,-50.1,-50.08
12,Sabah,Kudat,2019,Construction,22.73,11.14,-51.0,-50.97
22,Terengganu,Marang,2016,Construction,61.25,29.95,-51.1,-51.11
21,Terengganu,Dungun,2019,Construction,50.89,24.78,-51.3,-51.31
19,Sarawak,Pakan,2019,Construction,12.98,6.32,-51.4,-51.35
15,Sabah,Tawau,2020,Construction,231.17,111.03,-52.0,-51.97
3,Kedah,Kulim,2018,Construction,236.21,111.77,-52.7,-52.68


In [16]:
abs_gdp = gdp[gdp["series"] == "abs"][
    ["state", "district", "year", "sector", "value"]
].copy()

growth_check = growth.merge(
    abs_gdp,
    on=["state", "district", "year", "sector"],
    how="left"
).rename(columns={"value_x": "published_growth",
                  "value_y": "current_gdp"})

previous_gdp = abs_gdp.copy()
previous_gdp["year"] += 1
previous_gdp = previous_gdp.rename(
    columns={"value": "previous_gdp"}
)

growth_check = growth_check.merge(
    previous_gdp,
    on=["state", "district", "year", "sector"],
    how="left"
)

valid = (
    growth_check["previous_gdp"].notna() &
    growth_check["current_gdp"].notna() &
    (growth_check["previous_gdp"] != 0)
)

growth_check["calculated_growth"] = np.nan

growth_check.loc[valid, "calculated_growth"] = (
    (growth_check.loc[valid, "current_gdp"] /
     growth_check.loc[valid, "previous_gdp"] - 1) * 100
)

growth_check["difference"] = (
    growth_check["published_growth"] -
    growth_check["calculated_growth"]
).abs()

print("Total growth records:", len(growth_check))
print("Records with valid GDP comparisons:", valid.sum())
print("Records with difference above 0.15 percentage points:",
      (growth_check.loc[valid, "difference"] > 0.15).sum())

display(
    growth_check.loc[valid, "difference"].describe().round(6)
)

Total growth records: 4830
Records with valid GDP comparisons: 4099
Records with difference above 0.15 percentage points: 0


count    4099.000000
mean        0.025286
std         0.014321
min         0.000000
25%         0.013152
50%         0.024996
75%         0.037621
max         0.049993
Name: difference, dtype: float64

In [17]:
missing_gdp = gdp[gdp["value"].isna()].copy()

print("Total missing observations:", len(missing_gdp))

print("\nMissing values by series:")
display(missing_gdp["series"].value_counts())

print("\nMissing values by sector:")
display(
    missing_gdp.groupby(["sector", "desc_en"])
    .size()
    .reset_index(name="missing_count")
    .sort_values("missing_count", ascending=False)
)

print("\nTop 20 districts with missing GDP observations:")
display(
    missing_gdp.groupby(["state", "district"])
    .size()
    .reset_index(name="missing_count")
    .sort_values("missing_count", ascending=False)
    .head(20)
)

Total missing observations: 1238

Missing values by series:


series
growth_yoy    731
abs           507
Name: count, dtype: int64


Missing values by sector:


,sector,desc_en,missing_count
1,p2,Mining and quarrying,606
3,p4,Construction,365
2,p3,Manufacturing,231
0,p1,Agriculture,26
4,p5,Services,10



Top 20 districts with missing GDP observations:


,state,district,missing_count
75,Sarawak,Selangau,27
60,Sarawak,Julau,27
49,Sabah,Tambunan,27
58,Sarawak,Dalat,27
85,Sarawak,Telang Usan,27
84,Sarawak,Tebedu,27
69,Sarawak,Matu,27
71,Sarawak,Pusa,27
61,Sarawak,Kabong,27
41,Sabah,Nabawan,27


In [18]:
missing_pattern = (
    gdp.assign(is_missing=gdp["value"].isna())
    .pivot_table(
        index="year",
        columns=["series", "desc_en"],
        values="is_missing",
        aggfunc="sum",
        fill_value=0
    )
)

display(missing_pattern)

print("\nTotal missing values by year:")
display(
    gdp[gdp["value"].isna()]
    .groupby("year")
    .size()
    .rename("missing_count")
    .to_frame()
)

series          abs                                                       \
desc_en Agriculture Construction GDP at purchasers' prices Manufacturing   
year                                                                       
2015              1           27                         0            17   
2016              1           28                         0            17   
2017              1           30                         0            17   
2018              1           32                         0            17   
2019              1           34                         0            16   
2020              1           36                         0            17   

series                                 growth_yoy               \
desc_en Mining and quarrying Services Agriculture Construction   
year                                                             
2015                      39        0           0            0   
2016                      36        0           4           32   
2017                      34        0           4           34   
2018                      35        0           4           36   
2019                      34        0           4           37   
2020                      35        0           4           39   

series                                                                         
desc_en GDP at purchasers' prices Manufacturing Mining and quarrying Services  
year                                                                           
2015                            0             0                    0        0  
2016                            0            26                   83        2  
2017                            0            26                   78        2  
2018                            0            26                   77        2  
2019                            0            26                   77        2  
2020                            0            26                   78        2


Total missing values by year:


,missing_count
year,
2015,84
2016,229
2017,226
2018,230
2019,231
2020,238


In [19]:
print("GDP metadata columns:")
print(gdp_lookup.columns.tolist())

print("\nGDP lookup entries related to units and methodology:")
display(
    gdp_lookup[
        gdp_lookup.astype(str)
        .apply(
            lambda col: col.str.contains(
                "million|constant|2015|growth|purchaser",
                case=False,
                na=False
            )
        ).any(axis=1)
    ]
)

GDP metadata columns:
['method', 'code', 'desc_en', 'desc_bm']

GDP lookup entries related to units and methodology:


,method,code,desc_en,desc_bm
0,production,p0,GDP at purchasers' prices,KDNK pada harga pembeli
113,expenditure,e0,GDP at purchasers' prices,KDNK pada harga pembeli
168,income,i0,GDP at purchasers' prices,KDNK pada harga pembeli


In [20]:
print("Unique states:", gdp["state"].nunique())
print("Unique districts:", gdp["district"].nunique())

print("\nSpecial or potentially non-standard district records:")
display(
    gdp[
        gdp["district"].str.contains(
            "supra|putrajaya|kuala lumpur",
            case=False,
            na=False
        )
    ][["state", "district"]]
    .drop_duplicates()
    .sort_values(["state", "district"])
)

Unique states: 15
Unique districts: 160

Special or potentially non-standard district records:


,state,district
3384,Sabah,Supra
4932,Sarawak,Supra
5724,W.P. Kuala Lumpur,W.P. Kuala Lumpur


In [21]:
gdp_geo = gdp[["state", "district"]].drop_duplicates().copy()

gdp_geo["state_clean"] = (
    gdp_geo["state"].str.lower().str.strip()
)

gdp_geo["district_clean"] = (
    gdp_geo["district"].str.lower().str.strip()
)

lookup_geo = geo_lookup[
    ["state_clean", "district_clean", "code_state_district"]
].drop_duplicates()

gdp_geo_check = gdp_geo.merge(
    lookup_geo,
    on=["state_clean", "district_clean"],
    how="left",
    indicator=True
)

print("Total GDP geographic units:", len(gdp_geo_check))
print("Matched to official lookup:",
      (gdp_geo_check["_merge"] == "both").sum())
print("Unmatched geographic units:",
      (gdp_geo_check["_merge"] == "left_only").sum())

print("\nUnmatched records:")
display(
    gdp_geo_check[
        gdp_geo_check["_merge"] == "left_only"
    ][["state", "district"]]
    .sort_values(["state", "district"])
)

NameError: name 'geo_lookup' is not defined

In [22]:
census = pd.read_csv(CSV_DIR / "census_district.csv")

geo_lookup = (
    census[
        [
            "state",
            "code_state",
            "district",
            "code_district",
            "code_state_district"
        ]
    ]
    .drop_duplicates()
    .copy()
)

geo_lookup["state_clean"] = (
    geo_lookup["state"].str.lower().str.strip()
)

geo_lookup["district_clean"] = (
    geo_lookup["district"].str.lower().str.strip()
)

geo_lookup = geo_lookup.drop_duplicates(
    subset=["state_clean", "district_clean"]
)

print("Geographic lookup shape:", geo_lookup.shape)
print("Unique geographic units:",
      geo_lookup["code_state_district"].nunique())

Geographic lookup shape: (160, 7)
Unique geographic units: 160


In [23]:
gdp_geo = gdp[["state", "district"]].drop_duplicates().copy()

gdp_geo["state_clean"] = (
    gdp_geo["state"].str.lower().str.strip()
)

gdp_geo["district_clean"] = (
    gdp_geo["district"].str.lower().str.strip()
)

lookup_geo = geo_lookup[
    ["state_clean", "district_clean", "code_state_district"]
].drop_duplicates()

gdp_geo_check = gdp_geo.merge(
    lookup_geo,
    on=["state_clean", "district_clean"],
    how="left",
    indicator=True
)

print("Total GDP geographic units:", len(gdp_geo_check))
print("Matched to official lookup:",
      (gdp_geo_check["_merge"] == "both").sum())
print("Unmatched geographic units:",
      (gdp_geo_check["_merge"] == "left_only").sum())

print("\nUnmatched records:")
display(
    gdp_geo_check[
        gdp_geo_check["_merge"] == "left_only"
    ][["state", "district"]]
    .sort_values(["state", "district"])
)

Total GDP geographic units: 161
Matched to official lookup: 159
Unmatched geographic units: 2

Unmatched records:


,state,district
94,Sabah,Supra
137,Sarawak,Supra


In [24]:
gdp_matched = gdp_geo_check[
    gdp_geo_check["_merge"] == "both"
].copy()

lookup_not_in_gdp = lookup_geo.merge(
    gdp_matched[
        ["state_clean", "district_clean"]
    ],
    on=["state_clean", "district_clean"],
    how="left",
    indicator=True
)

print("Official districts absent from GDP:")
display(
    lookup_not_in_gdp[
        lookup_not_in_gdp["_merge"] == "left_only"
    ][
        ["state_clean", "district_clean", "code_state_district"]
    ]
)

Official districts absent from GDP:


,state_clean,district_clean,code_state_district
159,w.p. putrajaya,w.p. putrajaya,16_1


In [25]:
gdp_enriched = gdp.merge(
    lookup_geo[
        [
            "state_clean",
            "district_clean",
            "code_state",
            "code_district",
            "code_state_district"
        ]
    ],
    on=["state_clean", "district_clean"],
    how="left",
    validate="many_to_one"
)

print("Original GDP shape:", gdp.shape)
print("Enriched GDP shape:", gdp_enriched.shape)

print("\nMatched records:",
      gdp_enriched["code_state_district"].notna().sum())

print("Unmatched records:",
      gdp_enriched["code_state_district"].isna().sum())

print("\nUnmatched geographic records:")
display(
    gdp_enriched[
        gdp_enriched["code_state_district"].isna()
    ][["state", "district"]]
    .drop_duplicates()
)

KeyError: "['code_state', 'code_district'] not in index"

In [26]:
lookup_geo = geo_lookup[
    [
        "state_clean",
        "district_clean",
        "code_state",
        "code_district",
        "code_state_district"
    ]
].drop_duplicates()

gdp_enriched = gdp.merge(
    lookup_geo,
    on=["state_clean", "district_clean"],
    how="left",
    validate="many_to_one"
)

print("Original GDP shape:", gdp.shape)
print("Enriched GDP shape:", gdp_enriched.shape)

print("\nMatched records:",
      gdp_enriched["code_state_district"].notna().sum())

print("Unmatched records:",
      gdp_enriched["code_state_district"].isna().sum())

display(
    gdp_enriched[
        gdp_enriched["code_state_district"].isna()
    ][["state", "district"]]
    .drop_duplicates()
)

KeyError: 'state_clean'

In [27]:
gdp["state_clean"] = (
    gdp["state"].str.lower().str.strip()
)

gdp["district_clean"] = (
    gdp["district"].str.lower().str.strip()
)

gdp_enriched = gdp.merge(
    lookup_geo,
    on=["state_clean", "district_clean"],
    how="left",
    validate="many_to_one"
)

print("Original GDP records:", len(gdp))
print("Enriched GDP records:", len(gdp_enriched))

print("\nMatched records:",
      gdp_enriched["code_state_district"].notna().sum())

print("Unmatched records:",
      gdp_enriched["code_state_district"].isna().sum())

print("\nUnmatched geographic records:")
display(
    gdp_enriched[
        gdp_enriched["code_state_district"].isna()
    ][["state", "district"]]
    .drop_duplicates()
)

Original GDP records: 10626
Enriched GDP records: 10626

Matched records: 10494
Unmatched records: 132

Unmatched geographic records:


,state,district
3384,Sabah,Supra
4932,Sarawak,Supra


In [28]:
print("GDP enriched dataset validation")

print("\nShape:", gdp_enriched.shape)

print("\nDuplicate GDP records:",
      gdp_enriched.duplicated(
          subset=["series", "state", "district", "date", "sector"]
      ).sum())

print("\nMissing geographic codes:")
print(gdp_enriched["code_state_district"].isna().sum())

print("\nMissing GDP values:")
print(gdp_enriched["value"].isna().sum())

print("\nAvailable GDP years:")
print(sorted(gdp_enriched["year"].unique()))

print("\nUnique matched districts:",
      gdp_enriched["code_state_district"].nunique())

GDP enriched dataset validation

Shape: (10626, 14)

Duplicate GDP records: 0

Missing geographic codes:
132

Missing GDP values:
1238

Available GDP years:
[np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020)]

Unique matched districts: 159


In [29]:
output_dir = PROJECT_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

gdp_enriched.to_csv(
    output_dir / "KAWASAN_GDP_District_2015_2020.csv",
    index=False
)

gdp_enriched.to_excel(
    output_dir / "KAWASAN_GDP_District_2015_2020.xlsx",
    index=False
)

gdp_unmatched = gdp_enriched[
    gdp_enriched["code_state_district"].isna()
].copy()

gdp_unmatched.to_csv(
    output_dir / "KAWASAN_GDP_Unmatched_Geography.csv",
    index=False
)

print("GDP datasets exported successfully.")
print("Output directory:", output_dir)

GDP datasets exported successfully.
Output directory: c:\Users\Muhd Haziq\KAWASAN\data\processed


In [1]:
import pandas as pd
from pathlib import Path

print("KAWASAN environment ready!")
print("Current directory:", Path.cwd())

KAWASAN environment ready!
Current directory: c:\Users\Muhd Haziq\Desktop\KAWASAN\notebooks


In [2]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

kawasan_master_2024 = pd.read_csv(
    PROCESSED_DIR / "KAWASAN_Unified_District_Dataset_2024.csv"
)

gdp_enriched = pd.read_csv(
    PROCESSED_DIR / "KAWASAN_GDP_District_2015_2020.csv"
)

print("KAWASAN Unified Dataset:", kawasan_master_2024.shape)
print("District GDP Dataset:", gdp_enriched.shape)

print("\nUnified Dataset Columns:")
print(kawasan_master_2024.columns.tolist())

KAWASAN Unified Dataset: (155, 32)
District GDP Dataset: (10626, 14)

Unified Dataset Columns:
['state_clean', 'district_clean', 'code_state', 'code_district', 'code_state_district', 'income_mean', 'income_median', 'gini', 'poverty_absolute', 'poverty_relative', 'piped_water', 'sanitation', 'electricity', 'population', 'enrolment_primary', 'enrolment_secondary', 'enrolment_post_secondary', 'completion_primary', 'completion_secondary_lower', 'completion_secondary_upper', 'schools_primary', 'schools_secondary', 'schools_tertiary', 'teachers_primary', 'teachers_secondary', 'lf', 'lf_employed', 'lf_unemployed', 'lf_outside', 'p_rate', 'u_rate', 'ep_ratio']


In [3]:
integrated_datasets = {
    "hh_income_district",
    "hh_inequality_district",
    "hh_poverty_district",
    "hh_access_amenities",
    "population_district",
    "enrolment_school_district",
    "completion_school_district",
    "schools_district",
    "teachers_district",
    "lfs_district"
}

csv_files = sorted(CSV_DIR.glob("*.csv"))

if not csv_files:
    CSV_DIR = PROJECT_DIR / "data" / "raw"
    csv_files = sorted(CSV_DIR.rglob("*.csv"))

remaining = [
    f.stem for f in csv_files
    if f.stem not in integrated_datasets
]

print("Total CSV datasets:", len(csv_files))
print("Integrated into 2024 unified dataset:", len(integrated_datasets))
print("Remaining datasets:", len(remaining))

for dataset in remaining:
    print("-", dataset)
    

NameError: name 'CSV_DIR' is not defined

In [4]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA = PROJECT_DIR / "data" / "raw"
CSV_DIR = RAW_DATA / "csv"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("CSV directory:", CSV_DIR)
print("Directory exists:", CSV_DIR.exists())

CSV directory: c:\Users\Muhd Haziq\Desktop\KAWASAN\data\raw\csv
Directory exists: True


In [5]:
integrated_datasets = {
    "hh_income_district",
    "hh_inequality_district",
    "hh_poverty_district",
    "hh_access_amenities",
    "population_district",
    "enrolment_school_district",
    "completion_school_district",
    "schools_district",
    "teachers_district",
    "lfs_district"
}

csv_files = sorted(CSV_DIR.glob("*.csv"))

if not csv_files:
    CSV_DIR = PROJECT_DIR / "data" / "raw"
    csv_files = sorted(CSV_DIR.rglob("*.csv"))

remaining = [
    f.stem for f in csv_files
    if f.stem not in integrated_datasets
]

print("Total CSV datasets:", len(csv_files))
print("Integrated into 2024 unified dataset:", len(integrated_datasets))
print("Remaining datasets:", len(remaining))

for dataset in remaining:
    print("-", dataset)

Total CSV datasets: 25
Integrated into 2024 unified dataset: 10
Remaining datasets: 15
- census_district
- cpi_2d_state
- formal_wages_state
- gdp_district_real_supply
- gdp_state_real_supply
- hh_access_internet
- hies_district
- icths_internet_strata
- lfs_qtr_state
- lookup_premise
- pricecatcher_2026-11
- productivity_state
- wages_state
- water_access
- water_consumption


In [6]:
internet = pd.read_csv(
    CSV_DIR / "hh_access_internet.csv"
)

print("Shape:", internet.shape)

print("\nColumns:")
print(internet.columns.tolist())

print("\nSample records:")
print(internet.head(10).to_string(index=False))

print("\nMissing values:")
print(internet.isna().sum())

Shape: (343, 4)

Columns:
['date', 'state', 'strata', 'proportion']

Sample records:
      date    state  strata  proportion
2019-01-01 Malaysia overall        89.6
2020-01-01 Malaysia overall        91.0
2021-01-01 Malaysia overall        94.9
2022-01-01 Malaysia overall        96.0
2023-01-01 Malaysia overall        96.4
2024-01-01 Malaysia overall        96.8
2025-01-01 Malaysia overall        97.1
2019-01-01 Malaysia   urban        92.1
2020-01-01 Malaysia   urban        93.4
2021-01-01 Malaysia   urban        96.8

Missing values:
date          0
state         0
strata        0
proportion    0
dtype: int64


In [7]:
print("States:")
print(sorted(internet["state"].unique()))

print("\nStrata:")
print(internet["strata"].value_counts())

print("\nYears:")
print(sorted(pd.to_datetime(internet["date"]).dt.year.unique()))

print("\n2024 Internet Access:")
internet_2024 = internet[
    pd.to_datetime(internet["date"]).dt.year == 2024
]

print(
    internet_2024
    .sort_values(["state", "strata"])
    .to_string(index=False)
)

States:
['Johor', 'Kedah', 'Kelantan', 'Malaysia', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']

Strata:
strata
overall    119
urban      119
rural      105
Name: count, dtype: int64

Years:
[np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

2024 Internet Access:
      date             state  strata  proportion
2024-01-01             Johor overall        99.0
2024-01-01             Johor   rural        97.9
2024-01-01             Johor   urban        99.3
2024-01-01             Kedah overall        96.4
2024-01-01             Kedah   rural        93.5
2024-01-01             Kedah   urban        97.8
2024-01-01          Kelantan overall        94.9
2024-01-01          Kelantan   rural        94.1
2024-01-01          Kelantan   urban        96.0
2024-01-01          Malaysia overall        96.8
2024-

In [8]:
internet_clean = internet.copy()

internet_clean["year"] = pd.to_datetime(
    internet_clean["date"]
).dt.year

internet_wide = internet_clean.pivot(
    index=["state", "year"],
    columns="strata",
    values="proportion"
).reset_index()

internet_wide.columns.name = None

internet_wide["urban_rural_gap"] = (
    internet_wide["urban"] - internet_wide["rural"]
)

print("Shape:", internet_wide.shape)
print("\nDuplicate state-year records:",
      internet_wide.duplicated(["state", "year"]).sum())

print("\nSample:")
print(internet_wide.head(10).to_string(index=False))

Shape: (119, 6)

Duplicate state-year records: 0

Sample:
state  year  overall  rural  urban  urban_rural_gap
Johor  2019     94.1   89.9   95.1              5.2
Johor  2020     92.9   89.5   93.7              4.2
Johor  2021     98.0   95.7   98.5              2.8
Johor  2022     98.5   97.3   98.9              1.6
Johor  2023     98.8   97.4   99.1              1.7
Johor  2024     99.0   97.9   99.3              1.4
Johor  2025     99.0   98.0   99.3              1.3
Kedah  2019     82.6   77.7   84.3              6.6
Kedah  2020     90.8   86.9   92.1              5.2
Kedah  2021     95.1   91.5   96.3              4.8


In [9]:
print("Missing values:")
print(internet_wide.isna().sum())

print("\nValues outside 0–100%:")
percentage_cols = ["overall", "urban", "rural"]

for col in percentage_cols:
    invalid = (
        (internet_wide[col] < 0) |
        (internet_wide[col] > 100)
    ).sum()
    print(f"{col}: {invalid}")

print("\nStates with missing rural data:")
print(
    internet_wide.loc[
        internet_wide["rural"].isna(),
        ["state", "year"]
    ].to_string(index=False)
)

print("\nYear coverage:")
print(internet_wide.groupby("year")["state"].nunique())

Missing values:
state               0
year                0
overall             0
rural              14
urban               0
urban_rural_gap    14
dtype: int64

Values outside 0–100%:
overall: 0
urban: 0
rural: 0

States with missing rural data:
            state  year
W.P. Kuala Lumpur  2019
W.P. Kuala Lumpur  2020
W.P. Kuala Lumpur  2021
W.P. Kuala Lumpur  2022
W.P. Kuala Lumpur  2023
W.P. Kuala Lumpur  2024
W.P. Kuala Lumpur  2025
   W.P. Putrajaya  2019
   W.P. Putrajaya  2020
   W.P. Putrajaya  2021
   W.P. Putrajaya  2022
   W.P. Putrajaya  2023
   W.P. Putrajaya  2024
   W.P. Putrajaya  2025

Year coverage:
year
2019    17
2020    17
2021    17
2022    17
2023    17
2024    17
2025    17
Name: state, dtype: int64


In [10]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

internet_wide.to_csv(
    PROCESSED_DIR / "KAWASAN_Internet_Access_State_2019_2025.csv",
    index=False
)

internet_wide.to_excel(
    PROCESSED_DIR / "KAWASAN_Internet_Access_State_2019_2025.xlsx",
    index=False
)

print("Internet access dataset exported successfully!")
print("Records:", len(internet_wide))
print("Location:", PROCESSED_DIR)

Internet access dataset exported successfully!
Records: 119
Location: c:\Users\Muhd Haziq\Desktop\KAWASAN\data\processed


In [11]:
water = pd.read_csv(
    CSV_DIR / "water_access.csv"
)

print("Shape:", water.shape)

print("\nColumns:")
print(water.columns.tolist())

print("\nSample records:")
print(water.head(10).to_string(index=False))

print("\nMissing values:")
print(water.isna().sum())

print("\nYears:")
print(sorted(pd.to_datetime(water["date"]).dt.year.unique()))

Shape: (1161, 4)

Columns:
['state', 'strata', 'date', 'proportion']

Sample records:
   state  strata       date  proportion
Malaysia overall 2000-01-01        92.0
Malaysia overall 2001-01-01        92.7
Malaysia overall 2002-01-01        93.2
Malaysia overall 2003-01-01        93.7
Malaysia overall 2004-01-01        94.0
Malaysia overall 2005-01-01        95.0
Malaysia overall 2006-01-01        94.8
Malaysia overall 2007-01-01        90.5
Malaysia overall 2008-01-01        90.9
Malaysia overall 2009-01-01        93.0

Missing values:
state         0
strata        0
date          0
proportion    0
dtype: int64

Years:
[np.int32(2000), np.int32(2001), np.int32(2002), np.int32(2003), np.int32(2004), np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.

In [12]:
water["year"] = pd.to_datetime(water["date"]).dt.year

print("States:")
print(sorted(water["state"].unique()))

print("\nStrata:")
print(water["strata"].value_counts())

print("\nDuplicate state-year-strata records:")
print(water.duplicated(["state", "year", "strata"]).sum())

print("\nRecords per year:")
print(water.groupby("year").size())

print("\n2024 Water Access:")
print(
    water[water["year"] == 2024]
    .sort_values(["state", "strata"])
    .to_string(index=False)
)

States:
['Johor', 'Kedah', 'Kelantan', 'Malaysia', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Labuan']

Strata:
strata
overall    387
rural      387
urban      387
Name: count, dtype: int64

Duplicate state-year-strata records:
0

Records per year:
year
2000    45
2001    45
2002    45
2003    45
2004    45
2005    45
2006    45
2007    45
2008    45
2009    45
2010    45
2011    45
2012    45
2013    45
2014    45
2015    45
2016    45
2017    45
2018    45
2019    45
2020    45
2021    45
2022    45
2023    45
2024    45
2025    36
dtype: int64

2024 Water Access:
          state  strata       date  proportion  year
          Johor overall 2024-01-01        99.9  2024
          Johor   rural 2024-01-01        99.8  2024
          Johor   urban 2024-01-01       100.0  2024
          Kedah overall 2024-01-01        98.3  2024
          Kedah   rural 2024-01-01        96.5  2024
          Kedah   urban 20

In [13]:
states_2024 = set(water[water["year"] == 2024]["state"])
states_2025 = set(water[water["year"] == 2025]["state"])

print("States missing in 2025:")
print(sorted(states_2024 - states_2025))

print("\n2025 records:")
print(
    water[water["year"] == 2025]
    .sort_values(["state", "strata"])
    .to_string(index=False)
)

print("\nInvalid percentages:")
print(
    ((water["proportion"] < 0) |
     (water["proportion"] > 100)).sum()
)

States missing in 2025:
['Malaysia', 'Sabah', 'Sarawak']

2025 records:
          state  strata       date  proportion  year
          Johor overall 2025-01-01        99.9  2025
          Johor   rural 2025-01-01        99.8  2025
          Johor   urban 2025-01-01       100.0  2025
          Kedah overall 2025-01-01        98.3  2025
          Kedah   rural 2025-01-01        96.5  2025
          Kedah   urban 2025-01-01       100.0  2025
       Kelantan overall 2025-01-01        73.0  2025
       Kelantan   rural 2025-01-01        80.8  2025
       Kelantan   urban 2025-01-01        65.2  2025
         Melaka overall 2025-01-01       100.0  2025
         Melaka   rural 2025-01-01       100.0  2025
         Melaka   urban 2025-01-01       100.0  2025
Negeri Sembilan overall 2025-01-01        99.9  2025
Negeri Sembilan   rural 2025-01-01        99.8  2025
Negeri Sembilan   urban 2025-01-01       100.0  2025
         Pahang overall 2025-01-01        98.0  2025
         Pahang   rural 202

In [16]:
water_wide = water.pivot(
    index=["state", "year"],
    columns="strata",
    values="proportion"
).reset_index()

water_wide.columns.name = None

water_wide["urban_rural_gap"] = (
    water_wide["urban"] - water_wide["rural"]
)

water_wide["coverage_status"] = np.where(
    water_wide["year"] == 2025,
    "Partial geographic coverage",
    "Complete geographic coverage"
)

print("Shape:", water_wide.shape)

print("\nDuplicate state-year records:")
print(water_wide.duplicated(["state", "year"]).sum())

print("\nMissing values:")
print(water_wide.isna().sum())

print("\nSample:")
print(water_wide.head(10).to_string(index=False))

Shape: (387, 7)

Duplicate state-year records:
0

Missing values:
state              0
year               0
overall            0
rural              0
urban              0
urban_rural_gap    0
coverage_status    0
dtype: int64

Sample:
state  year  overall  rural  urban  urban_rural_gap              coverage_status
Johor  2000     99.0   98.0  100.0              2.0 Complete geographic coverage
Johor  2001     99.7   99.0  100.0              1.0 Complete geographic coverage
Johor  2002     99.6   99.0  100.0              1.0 Complete geographic coverage
Johor  2003     99.7   99.0  100.0              1.0 Complete geographic coverage
Johor  2004     99.4   98.0  100.0              2.0 Complete geographic coverage
Johor  2005     99.5   99.0  100.0              1.0 Complete geographic coverage
Johor  2006     99.5   99.0  100.0              1.0 Complete geographic coverage
Johor  2007     99.8   99.5  100.0              0.5 Complete geographic coverage
Johor  2008     99.8   99.5  100.0  

In [15]:
import numpy as np

In [17]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

water_wide.to_csv(
    PROCESSED_DIR / "KAWASAN_Water_Access_State_2000_2025.csv",
    index=False
)

water_wide.to_excel(
    PROCESSED_DIR / "KAWASAN_Water_Access_State_2000_2025.xlsx",
    index=False
)

print("Water access dataset exported successfully!")
print("Records:", len(water_wide))
print("Location:", PROCESSED_DIR)

Water access dataset exported successfully!
Records: 387
Location: c:\Users\Muhd Haziq\Desktop\KAWASAN\data\processed


In [18]:
water_consumption = pd.read_csv(
    CSV_DIR / "water_consumption.csv"
)

print("Shape:", water_consumption.shape)

print("\nColumns:")
print(water_consumption.columns.tolist())

print("\nSample records:")
print(water_consumption.head(10).to_string(index=False))

print("\nMissing values:")
print(water_consumption.isna().sum())

print("\nAvailable years:")
print(sorted(pd.to_datetime(water_consumption["date"]).dt.year.unique()))

Shape: (660, 4)

Columns:
['state', 'sector', 'date', 'value']

Sample records:
   state   sector       date  value
Malaysia domestic 2003-01-01   4394
Malaysia domestic 2004-01-01   4770
Malaysia domestic 2005-01-01   4992
Malaysia domestic 2006-01-01   5025
Malaysia domestic 2007-01-01   5381
Malaysia domestic 2008-01-01   5466
Malaysia domestic 2009-01-01   5531
Malaysia domestic 2010-01-01   5602
Malaysia domestic 2011-01-01   5695
Malaysia domestic 2012-01-01   5873

Missing values:
state     0
sector    0
date      0
value     0
dtype: int64

Available years:
[np.int32(2003), np.int32(2004), np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]


In [19]:
water_consumption["year"] = pd.to_datetime(
    water_consumption["date"]
).dt.year

print("States:")
print(sorted(water_consumption["state"].unique()))

print("\nSectors:")
print(water_consumption["sector"].value_counts())

print("\nDuplicate records:")
print(
    water_consumption.duplicated(
        ["state", "year", "sector"]
    ).sum()
)

print("\nRecords per year:")
print(water_consumption.groupby("year").size())

print("\n2024 Water Consumption:")
print(
    water_consumption[
        water_consumption["year"] == 2024
    ]
    .sort_values(["state", "sector"])
    .to_string(index=False)
)

States:
['Johor', 'Kedah', 'Kelantan', 'Malaysia', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Labuan']

Sectors:
sector
domestic       330
nondomestic    330
Name: count, dtype: int64

Duplicate records:
0

Records per year:
year
2003    30
2004    30
2005    30
2006    30
2007    30
2008    30
2009    30
2010    30
2011    30
2012    30
2013    30
2014    30
2015    30
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
dtype: int64

2024 Water Consumption:
          state      sector       date  value  year
          Johor    domestic 2024-01-01    901  2024
          Johor nondomestic 2024-01-01    610  2024
          Kedah    domestic 2024-01-01    499  2024
          Kedah nondomestic 2024-01-01    283  2024
       Kelantan    domestic 2024-01-01    164  2024
       Kelantan nondomestic 2024-01-01     80  2024
       Malaysia    domestic 2024-01-01   75

In [20]:
consumption_wide = water_consumption.pivot(
    index=["state", "year"],
    columns="sector",
    values="value"
).reset_index()

consumption_wide.columns.name = None

consumption_wide["total_consumption"] = (
    consumption_wide["domestic"] +
    consumption_wide["nondomestic"]
)

consumption_wide["domestic_share"] = (
    consumption_wide["domestic"] /
    consumption_wide["total_consumption"] * 100
).where(consumption_wide["total_consumption"] > 0)

consumption_wide["nondomestic_share"] = (
    consumption_wide["nondomestic"] /
    consumption_wide["total_consumption"] * 100
).where(consumption_wide["total_consumption"] > 0)

print("Shape:", consumption_wide.shape)

print("\nDuplicate state-year records:")
print(consumption_wide.duplicated(["state", "year"]).sum())

print("\nMissing values:")
print(consumption_wide.isna().sum())

print("\nSample:")
print(consumption_wide.head(10).to_string(index=False))

Shape: (330, 7)

Duplicate state-year records:
0

Missing values:
state                0
year                 0
domestic             0
nondomestic          0
total_consumption    0
domestic_share       0
nondomestic_share    0
dtype: int64

Sample:
state  year  domestic  nondomestic  total_consumption  domestic_share  nondomestic_share
Johor  2003       560          246                806       69.478908          30.521092
Johor  2004       610          225                835       73.053892          26.946108
Johor  2005       621          235                856       72.546729          27.453271
Johor  2006       644          255                899       71.635150          28.364850
Johor  2007       664          289                953       69.674711          30.325289
Johor  2008       682          300                982       69.450102          30.549898
Johor  2009       713          285                998       71.442886          28.557114
Johor  2010       730          306     

In [21]:
print("Negative consumption values:")
print(
    (consumption_wide[
        ["domestic", "nondomestic", "total_consumption"]
    ] < 0).sum()
)

print("\nInvalid sector shares:")
print(
    (
        (consumption_wide[["domestic_share", "nondomestic_share"]] < 0) |
        (consumption_wide[["domestic_share", "nondomestic_share"]] > 100)
    ).sum()
)

print("\nShares not adding up to 100%:")
print(
    (~np.isclose(
        consumption_wide["domestic_share"] +
        consumption_wide["nondomestic_share"],
        100
    )).sum()
)

print("\nYears covered per state:")
print(consumption_wide.groupby("state")["year"].nunique())

Negative consumption values:
domestic             0
nondomestic          0
total_consumption    0
dtype: int64

Invalid sector shares:
domestic_share       0
nondomestic_share    0
dtype: int64

Shares not adding up to 100%:
0

Years covered per state:
state
Johor              22
Kedah              22
Kelantan           22
Malaysia           22
Melaka             22
Negeri Sembilan    22
Pahang             22
Perak              22
Perlis             22
Pulau Pinang       22
Sabah              22
Sarawak            22
Selangor           22
Terengganu         22
W.P. Labuan        22
Name: year, dtype: int64


In [22]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

consumption_wide.to_csv(
    PROCESSED_DIR / "KAWASAN_Water_Consumption_State_2003_2024.csv",
    index=False
)

consumption_wide.to_excel(
    PROCESSED_DIR / "KAWASAN_Water_Consumption_State_2003_2024.xlsx",
    index=False
)

print("Water consumption dataset exported successfully!")
print("Records:", len(consumption_wide))
print("Location:", PROCESSED_DIR)

Water consumption dataset exported successfully!
Records: 330
Location: c:\Users\Muhd Haziq\Desktop\KAWASAN\data\processed


In [23]:
import pandas as pd
from pathlib import Path

completed = {
    "census_district",
    "gdp_district_real_supply",
    "hh_access_internet",
    "water_access",
    "water_consumption"
}

datasets_to_audit = [
    f for f in csv_files
    if f.stem in remaining and f.stem not in completed
]

audit_results = []

for file in datasets_to_audit:
    df = pd.read_csv(file, low_memory=False)

    date_col = next(
        (c for c in ["date", "year"] if c in df.columns),
        None
    )

    if date_col == "date":
        years = pd.to_datetime(
            df["date"], errors="coerce"
        ).dt.year.dropna()

    elif date_col == "year":
        years = pd.to_numeric(
            df["year"], errors="coerce"
        ).dropna()

    else:
        years = pd.Series(dtype=float)

    if "district" in df.columns:
        geography = "District"
    elif "state" in df.columns:
        geography = "State"
    elif "premise_code" in df.columns:
        geography = "Premise"
    else:
        geography = "Other / Unknown"

    audit_results.append({
        "dataset": file.stem,
        "rows": len(df),
        "columns": len(df.columns),
        "geography": geography,
        "year_start": int(years.min()) if len(years) else None,
        "year_end": int(years.max()) if len(years) else None,
        "missing_values": int(df.isna().sum().sum()),
        "column_names": ", ".join(df.columns)
    })

audit_summary = pd.DataFrame(audit_results)

print("Datasets audited:", len(audit_summary))

display(
    audit_summary.drop(columns=["column_names"])
)

print("\nDataset columns:")
for _, row in audit_summary.iterrows():
    print(f"\n{row['dataset']}")
    print(row["column_names"])

Datasets audited: 10


,dataset,rows,columns,geography,year_start,year_end,missing_values
0,cpi_2d_state,45024,4,State,2010.0,2026.0,0
1,formal_wages_state,918,4,State,2022.0,2026.0,0
2,gdp_state_real_supply,2163,5,State,2015.0,2025.0,1
3,hies_district,322,8,District,2022.0,2024.0,0
4,icths_internet_strata,392,4,State,2018.0,2025.0,0
5,lfs_qtr_state,624,8,State,2017.0,2026.0,0
6,lookup_premise,3932,6,District,NaN,NaN,3
7,pricecatcher_2026-11,797250,4,Premise,2026.0,2026.0,0
8,productivity_state,2142,5,State,2015.0,2025.0,0
9,wages_state,816,6,State,2010.0,2025.0,0



Dataset columns:

cpi_2d_state
state, date, division, index

formal_wages_state
state, date, employees, wages_median

gdp_state_real_supply
series, date, state, sector, value

hies_district
date, state, district, income_mean, income_median, expenditure_mean, gini, poverty

icths_internet_strata
date, state, strata, proportion

lfs_qtr_state
date, state, lf, lf_employed, lf_unemployed, lf_outside, p_rate, u_rate

lookup_premise
premise_code, premise, address, premise_type, state, district

pricecatcher_2026-11
date, premise_code, item_code, price

productivity_state
series, date, state, sector, output_employment

wages_state
state, date, sex, recipients, wages_median, wages_mean


In [24]:
for name in [
    "hies_district",
    "lookup_premise",
    "pricecatcher_2026-11"
]:
    df = pd.read_csv(CSV_DIR / f"{name}.csv")

    print(f"\n{'=' * 50}")
    print(name.upper())
    print("Shape:", df.shape)
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

    if "date" in df.columns:
        dates = pd.to_datetime(df["date"], errors="coerce")
        print("Date range:", dates.min(), "to", dates.max())

    if "district" in df.columns:
        print(
            "Unique state-district pairs:",
            df[["state", "district"]].drop_duplicates().shape[0]
        )

    if "premise_code" in df.columns:
        print("Unique premises:", df["premise_code"].nunique())

    if "item_code" in df.columns:
        print("Unique items:", df["item_code"].nunique())

    print("\nSample:")
    print(df.head(3).to_string(index=False))


HIES_DISTRICT
Shape: (322, 8)
Missing values: 0
Duplicate rows: 0
Date range: 2022-01-01 00:00:00 to 2024-01-01 00:00:00
Unique state-district pairs: 172

Sample:
      date state    district  income_mean  income_median  expenditure_mean  gini  poverty
2022-01-01 Johor  Batu Pahat         7419           6347              4570 0.338      5.1
2024-01-01 Johor  Batu Pahat         9006           7555              5153 0.316      2.4
2022-01-01 Johor Johor Bahru         9869           8232              6139 0.358      3.7

LOOKUP_PREMISE
Shape: (3932, 6)
Missing values: 3
Duplicate rows: 0
Unique state-district pairs: 179
Unique premises: 3932

Sample:
 premise_code          premise                                                  address             premise_type  state      district
           -1  Unknown Premise                                          Unknown Address                      NaN    NaN           NaN
            2 PASAR BESAR IPOH          JALAN LAKSAMANA,TAMAN JUBILEE,30300

In [25]:
hies = pd.read_csv(CSV_DIR / "hies_district.csv")

hies["year"] = pd.to_datetime(hies["date"]).dt.year

hies_2024 = hies[hies["year"] == 2024].copy()

print("2024 HIES records:", len(hies_2024))

print("\nDuplicate state-district pairs:")
print(
    hies_2024.duplicated(["state", "district"]).sum()
)

print("\n2024 missing values:")
print(hies_2024.isna().sum())

print("\nExpenditure statistics:")
print(hies_2024["expenditure_mean"].describe())

print("\nSample:")
print(hies_2024.head().to_string(index=False))

2024 HIES records: 162

Duplicate state-district pairs:
0

2024 missing values:
date                0
state               0
district            0
income_mean         0
income_median       0
expenditure_mean    0
gini                0
poverty             0
year                0
dtype: int64

Expenditure statistics:
count     162.000000
mean     4046.148148
std      1140.112391
min      2257.000000
25%      3228.750000
50%      3758.500000
75%      4643.750000
max      8210.000000
Name: expenditure_mean, dtype: float64

Sample:
      date state    district  income_mean  income_median  expenditure_mean  gini  poverty  year
2024-01-01 Johor  Batu Pahat         9006           7555              5153 0.316      2.4  2024
2024-01-01 Johor Johor Bahru        10975           8977              6764 0.350      1.9  2024
2024-01-01 Johor      Kluang         6616           6158              4748 0.292      5.7  2024
2024-01-01 Johor Kota Tinggi         8106           6284              4949 0.379    

In [26]:
hies_2024["state_clean"] = (
    hies_2024["state"].str.lower().str.strip()
)

hies_2024["district_clean"] = (
    hies_2024["district"].str.lower().str.strip()
)

hies_check = kawasan_master_2024.merge(
    hies_2024[
        [
            "state_clean",
            "district_clean",
            "income_mean",
            "income_median",
            "gini",
            "poverty",
            "expenditure_mean"
        ]
    ],
    on=["state_clean", "district_clean"],
    how="left",
    suffixes=("_master", "_hies"),
    indicator=True,
    validate="one_to_one"
)

print("Total master districts:", len(hies_check))

print("\nGeographic matching:")
print(hies_check["_merge"].value_counts())

print("\nIncome comparison:")
for col in ["income_mean", "income_median", "gini"]:
    differences = ~np.isclose(
        hies_check[f"{col}_master"],
        hies_check[f"{col}_hies"],
        equal_nan=True
    )

    matched = hies_check["_merge"] == "both"

    print(f"{col}: {(differences & matched).sum()} mismatches")

print("\nDistricts without HIES match:")
print(
    hies_check.loc[
        hies_check["_merge"] == "left_only",
        ["state_clean", "district_clean"]
    ].to_string(index=False)
)

Total master districts: 155

Geographic matching:
_merge
both          150
left_only       5
right_only      0
Name: count, dtype: int64

Income comparison:
income_mean: 0 mismatches
income_median: 0 mismatches
gini: 0 mismatches

Districts without HIES match:
 state_clean         district_clean
pulau pinang  seberang perai tengah
pulau pinang   seberang perai utara
pulau pinang seberang perai selatan
    selangor            hulu langat
    selangor          hulu selangor


In [27]:
print(
    hies_2024[
        hies_2024["state"].isin(["Pulau Pinang", "Selangor"])
    ][["state", "district"]]
    .sort_values(["state", "district"])
    .to_string(index=False)
)

       state       district
Pulau Pinang     Barat Daya
Pulau Pinang   S.P. Selatan
Pulau Pinang     S.P.Tengah
Pulau Pinang      S.P.Utara
Pulau Pinang     Timur Laut
    Selangor         Gombak
    Selangor          Klang
    Selangor   Kuala Langat
    Selangor Kuala Selangor
    Selangor       Petaling
    Selangor   Sabak Bernam
    Selangor         Sepang
    Selangor     Ulu Langat
    Selangor   Ulu Selangor


In [28]:
district_aliases = {
    "s.p. selatan": "seberang perai selatan",
    "s.p.tengah": "seberang perai tengah",
    "s.p.utara": "seberang perai utara",
    "ulu langat": "hulu langat",
    "ulu selangor": "hulu selangor"
}

hies_2024["district_clean"] = (
    hies_2024["district_clean"]
    .replace(district_aliases)
)

hies_matched = kawasan_master_2024.merge(
    hies_2024[
        ["state_clean", "district_clean",
         "income_mean", "income_median",
         "gini", "poverty", "expenditure_mean"]
    ],
    on=["state_clean", "district_clean"],
    how="left",
    suffixes=("_master", "_hies"),
    indicator=True,
    validate="one_to_one"
)

print("Geographic matching:")
print(hies_matched["_merge"].value_counts())

print("\nMissing expenditure values:")
print(hies_matched["expenditure_mean"].isna().sum())

print("\nIndicator mismatches:")
for col in ["income_mean", "income_median", "gini"]:
    mismatches = ~np.isclose(
        hies_matched[f"{col}_master"],
        hies_matched[f"{col}_hies"],
        equal_nan=True
    )
    print(f"{col}: {mismatches.sum()}")

print("\nAbsolute poverty mismatches:")
print(
    (~np.isclose(
        hies_matched["poverty_absolute"],
        hies_matched["poverty"],
        equal_nan=True
    )).sum()
)

Geographic matching:
_merge
both          155
left_only       0
right_only      0
Name: count, dtype: int64

Missing expenditure values:
0

Indicator mismatches:
income_mean: 0
income_median: 0
gini: 0

Absolute poverty mismatches:
0


In [29]:
# Add household expenditure to the unified dataset
kawasan_master_2024["expenditure_mean"] = (
    kawasan_master_2024.merge(
        hies_2024[
            ["state_clean", "district_clean", "expenditure_mean"]
        ],
        on=["state_clean", "district_clean"],
        how="left",
        validate="one_to_one"
    )["expenditure_mean"].to_numpy()
)

# Final validation
print("Shape:", kawasan_master_2024.shape)
print("Unique districts:", kawasan_master_2024["code_state_district"].nunique())
print("Missing expenditure:", kawasan_master_2024["expenditure_mean"].isna().sum())
print("Duplicate district codes:",
      kawasan_master_2024["code_state_district"].duplicated().sum())

# Export updated datasets
kawasan_master_2024.to_csv(
    PROCESSED_DIR / "KAWASAN_Unified_District_Dataset_2024.csv",
    index=False
)

kawasan_master_2024.to_excel(
    PROCESSED_DIR / "KAWASAN_Unified_District_Dataset_2024.xlsx",
    index=False
)

print("\nUpdated KAWASAN Unified Dataset exported!")

Shape: (155, 33)
Unique districts: 155
Missing expenditure: 0
Duplicate district codes: 0

Updated KAWASAN Unified Dataset exported!


In [30]:
state_datasets = [
    "cpi_2d_state",
    "formal_wages_state",
    "gdp_state_real_supply",
    "hh_access_internet",
    "icths_internet_strata",
    "lfs_qtr_state",
    "productivity_state",
    "wages_state",
    "water_access",
    "water_consumption"
]

results = []

for name in state_datasets:
    df = pd.read_csv(CSV_DIR / f"{name}.csv", low_memory=False)

    dates = pd.to_datetime(df["date"], errors="coerce")
    years = dates.dt.year

    results.append({
        "Dataset": name,
        "Rows": len(df),
        "States": df["state"].nunique(),
        "Start Year": int(years.min()),
        "End Year": int(years.max()),
        "Unique Dates": dates.nunique(),
        "Rows per Date": round(len(df) / dates.nunique(), 1),
        "Missing Values": int(df.isna().sum().sum())
    })

state_audit = pd.DataFrame(results)

display(state_audit)

,Dataset,Rows,States,Start Year,End Year,Unique Dates,Rows per Date,Missing Values
0,cpi_2d_state,45024,16,2010,2026,201,224.0,0
1,formal_wages_state,918,17,2022,2026,54,17.0,0
2,gdp_state_real_supply,2163,17,2015,2025,11,196.6,1
3,hh_access_internet,343,17,2019,2025,7,49.0,0
4,icths_internet_strata,392,17,2018,2025,8,49.0,0
5,lfs_qtr_state,624,16,2017,2026,39,16.0,0
6,productivity_state,2142,18,2015,2025,11,194.7,0
7,wages_state,816,17,2010,2025,16,51.0,0
8,water_access,1161,15,2000,2025,26,44.7,0
9,water_consumption,660,15,2003,2024,22,30.0,0


In [31]:
for name in state_datasets:
    df = pd.read_csv(CSV_DIR / f"{name}.csv", low_memory=False)

    print(f"\n{'=' * 55}")
    print(name.upper())

    print("\nGeographic categories:")
    print(sorted(df["state"].dropna().unique().tolist()))

    for col in ["series", "sector", "strata", "sex", "division"]:
        if col in df.columns:
            values = df[col].dropna().unique()

            print(f"\n{col}: {len(values)} categories")
            print(values[:25])

    dates = pd.to_datetime(df["date"], errors="coerce")

    print("\nDate examples:")
    print(
        sorted(dates.dropna().dt.strftime("%Y-%m-%d").unique())[-5:]
    )


CPI_2D_STATE

Geographic categories:
['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']

division: 14 categories
<StringArray>
['overall',      '01',      '02',      '03',      '04',      '05',      '06',
      '07',      '08',      '09',      '10',      '11',      '12',      '13']
Length: 14, dtype: str

Date examples:
['2026-05-01', '2026-06-01', '2026-07-01', '2026-08-01', '2026-09-01']

FORMAL_WAGES_STATE

Geographic categories:
['Johor', 'Kedah', 'Kelantan', 'Malaysia', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']

Date examples:
['2026-02-01', '2026-03-01', '2026-04-01', '2026-05-01', '2026-06-01']

GDP_STATE_REAL_SUPPLY

Geographic categories:
['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan

In [32]:
import pandas as pd

states = sorted(
    kawasan_master_2024["state_clean"]
    .dropna()
    .unique()
)

years = range(2010, 2026)

state_master = pd.MultiIndex.from_product(
    [states, years],
    names=["state_clean", "year"]
).to_frame(index=False)

sources = [
    ("hh_access_internet", "strata", "overall",
     "proportion", "hh_internet_access"),

    ("icths_internet_strata", "strata", "overall",
     "proportion", "icths_internet"),

    ("water_access", "strata", "overall",
     "proportion", "water_access"),

    ("water_consumption", "sector", "domestic",
     "value", "water_consumption_domestic"),

    ("water_consumption", "sector", "nondomestic",
     "value", "water_consumption_nondomestic"),

    ("wages_state", "sex", "both",
     "wages_median", "wages_median"),

    ("wages_state", "sex", "both",
     "wages_mean", "wages_mean")
]

for filename, filter_col, filter_value, value_col, output_col in sources:

    df = pd.read_csv(CSV_DIR / f"{filename}.csv")

    df = df[df[filter_col] == filter_value].copy()

    df["year"] = pd.to_datetime(df["date"]).dt.year

    df["state_clean"] = (
        df["state"].str.lower().str.strip()
    )

    df = df[
        ["state_clean", "year", value_col]
    ].rename(columns={value_col: output_col})

    if df.duplicated(["state_clean", "year"]).any():
        raise ValueError(
            f"Duplicate state-year records: {filename}"
        )

    state_master = state_master.merge(
        df,
        on=["state_clean", "year"],
        how="left",
        validate="one_to_one"
    )

print("Shape:", state_master.shape)
print("States:", state_master["state_clean"].nunique())
print("Years:", state_master["year"].min(), "-", state_master["year"].max())
print("Duplicate state-year records:",
      state_master.duplicated(["state_clean", "year"]).sum())

print("\nIndicator coverage:")
print(state_master.notna().sum())

display(state_master.head(10))

Shape: (192, 9)
States: 12
Years: 2010 - 2025
Duplicate state-year records: 0

Indicator coverage:
state_clean                      192
year                             192
hh_internet_access                84
icths_internet                    96
water_access                     190
water_consumption_domestic       180
water_consumption_nondomestic    180
wages_median                     192
wages_mean                       192
dtype: int64


,state_clean,year,hh_internet_access,icths_internet,water_access,water_consumption_domestic,water_consumption_nondomestic,wages_median,wages_mean
0,johor,2010,NaN,NaN,99.8,730.0,306.0,1500,1795
1,johor,2011,NaN,NaN,99.8,738.0,323.0,1520,1940
2,johor,2012,NaN,NaN,99.8,769.0,341.0,1695,2035
3,johor,2013,NaN,NaN,99.8,797.0,366.0,1700,2114
4,johor,2014,NaN,NaN,99.8,823.0,391.0,2000,2466
5,johor,2015,NaN,NaN,99.8,811.0,448.0,2200,2549
6,johor,2016,NaN,NaN,99.8,773.0,513.0,2200,2656
7,johor,2017,NaN,NaN,99.8,785.0,535.0,2250,2932
8,johor,2018,NaN,83.2,99.8,809.0,576.0,2418,3228
9,johor,2019,94.1,89.6,99.8,835.0,599.0,2512,3266


In [33]:
# Use state categories from the CPI dataset
cpi_states = pd.read_csv(
    CSV_DIR / "cpi_2d_state.csv",
    usecols=["state"]
)

states = sorted(
    cpi_states["state"]
    .dropna()
    .str.lower()
    .str.strip()
    .unique()
)

years = range(2010, 2026)

state_master = pd.MultiIndex.from_product(
    [states, years],
    names=["state_clean", "year"]
).to_frame(index=False)

for filename, filter_col, filter_value, value_col, output_col in sources:

    df = pd.read_csv(CSV_DIR / f"{filename}.csv")

    df = df[df[filter_col] == filter_value].copy()

    df["year"] = pd.to_datetime(df["date"]).dt.year

    df["state_clean"] = (
        df["state"].str.lower().str.strip()
    )

    df = df[
        ["state_clean", "year", value_col]
    ].rename(columns={value_col: output_col})

    if df.duplicated(["state_clean", "year"]).any():
        raise ValueError(
            f"Duplicate state-year records: {filename}"
        )

    state_master = state_master.merge(
        df,
        on=["state_clean", "year"],
        how="left",
        validate="one_to_one"
    )

print("Shape:", state_master.shape)
print("Geographic categories:", state_master["state_clean"].nunique())
print("Years:", state_master["year"].min(), "-", state_master["year"].max())
print("Duplicate state-year records:",
      state_master.duplicated(["state_clean", "year"]).sum())

print("\nIndicator coverage:")
print(state_master.notna().sum())

print("\nGeographic categories:")
print(sorted(state_master["state_clean"].unique()))

Shape: (256, 9)
Geographic categories: 16
Years: 2010 - 2025
Duplicate state-year records: 0

Indicator coverage:
state_clean                      256
year                             256
hh_internet_access               112
icths_internet                   128
water_access                     222
water_consumption_domestic       210
water_consumption_nondomestic    210
wages_median                     256
wages_mean                       256
dtype: int64

Geographic categories:
['johor', 'kedah', 'kelantan', 'melaka', 'negeri sembilan', 'pahang', 'perak', 'perlis', 'pulau pinang', 'sabah', 'sarawak', 'selangor', 'terengganu', 'w.p. kuala lumpur', 'w.p. labuan', 'w.p. putrajaya']


In [35]:
economic_sources = [
    ("gdp_state_real_supply", "gdp_total"),
    ("productivity_state", "labour_productivity")
]

economic_prepared = {}

for filename, output_col in economic_sources:
    df = pd.read_csv(CSV_DIR / f"{filename}.csv")

    df = df[
        (df["series"] == "abs") &
        (df["sector"] == "p0")
    ].copy()

    df["year"] = pd.to_datetime(df["date"]).dt.year
    df["state_clean"] = df["state"].str.lower().str.strip()

    # Retain only the 16 state/federal territory categories
    df = df[df["state_clean"].isin(states)]

    df = df[
        ["state_clean", "year", "value" if filename == "gdp_state_real_supply"
         else "output_employment"]
    ].copy()

    value_col = (
        "value" if filename == "gdp_state_real_supply"
        else "output_employment"
    )

    df = df.rename(columns={value_col: output_col})

    economic_prepared[output_col] = df

    print(f"\n{filename}")
    print("Rows:", len(df))
    print("States:", df["state_clean"].nunique())
    print("Years:", df["year"].min(), "-", df["year"].max())
    print("Duplicate state-year records:",
          df.duplicated(["state_clean", "year"]).sum())
    print("Missing values:", df[output_col].isna().sum())

    display(df.head(3))


gdp_state_real_supply
Rows: 168
States: 16
Years: 2015 - 2025
Duplicate state-year records: 0
Missing values: 0


,state_clean,year,gdp_total
0,johor,2015,110002.221
7,kedah,2015,39549.581
14,kelantan,2015,21407.736



productivity_state
Rows: 176
States: 16
Years: 2015 - 2025
Duplicate state-year records: 0
Missing values: 0


,state_clean,year,labour_productivity
0,johor,2015,64381.0
6,kedah,2015,72534.0
12,kelantan,2015,60072.0


In [36]:
for indicator, df in economic_prepared.items():

    state_master = state_master.merge(
        df,
        on=["state_clean", "year"],
        how="left",
        validate="one_to_one"
    )

print("Updated shape:", state_master.shape)

print("Geographic categories:", state_master["state_clean"].nunique())

print("Duplicate state-year records:",
      state_master.duplicated(["state_clean", "year"]).sum())

print("\nGDP and productivity coverage:")
print(
    state_master[
        ["gdp_total", "labour_productivity"]
    ].notna().sum()
)

print("\nMissing GDP state-year combinations:")
print(
    state_master.loc[
        state_master["year"].between(2015, 2025)
        & state_master["gdp_total"].isna(),
        ["state_clean", "year"]
    ].to_string(index=False)
)

print("\n2024 sample:")
display(
    state_master[
        state_master["year"] == 2024
    ].head()
)


Updated shape: (256, 11)
Geographic categories: 16
Duplicate state-year records: 0

GDP and productivity coverage:
gdp_total              168
labour_productivity    176
dtype: int64

Missing GDP state-year combinations:
   state_clean  year
w.p. putrajaya  2015
w.p. putrajaya  2016
w.p. putrajaya  2017
w.p. putrajaya  2018
w.p. putrajaya  2019
w.p. putrajaya  2020
w.p. putrajaya  2021
w.p. putrajaya  2022

2024 sample:


,state_clean,year,hh_internet_access,icths_internet,water_access,water_consumption_domestic,water_consumption_nondomestic,wages_median,wages_mean,gdp_total,labour_productivity
14,johor,2024,99.0,98.9,99.9,901.0,610.0,2582,3414,158348.264,81214.0
30,kedah,2024,96.4,97.4,98.3,499.0,283.0,2051,3053,54101.506,83295.0
46,kelantan,2024,94.9,97.3,73.5,164.0,80.0,1914,3091,28599.018,71838.0
62,melaka,2024,96.8,98.5,100.0,242.0,215.0,2528,3469,48844.201,110840.0
78,negeri sembilan,2024,96.4,98.6,99.9,323.0,237.0,2503,3626,54335.264,108180.0


In [37]:
cpi = pd.read_csv(CSV_DIR / "cpi_2d_state.csv")

cpi = cpi[cpi["division"] == "overall"].copy()

cpi["date"] = pd.to_datetime(cpi["date"])
cpi["year"] = cpi["date"].dt.year
cpi["state_clean"] = cpi["state"].str.lower().str.strip()

cpi_annual = (
    cpi.groupby(["state_clean", "year"])
    .agg(
        cpi_annual_average=("index", "mean"),
        months_available=("date", "nunique")
    )
    .reset_index()
)

print("Annual CPI records:", len(cpi_annual))

print("\nMonthly coverage:")
print(
    cpi_annual.groupby("year")["months_available"]
    .agg(["min", "max"])
)

print("\nIncomplete state-years:")
display(
    cpi_annual[cpi_annual["months_available"] != 12]
)

print("\nDuplicate state-year records:",
      cpi_annual.duplicated(["state_clean", "year"]).sum())

Annual CPI records: 272

Monthly coverage:
      min  max
year          
2010   12   12
2011   12   12
2012   12   12
2013   12   12
2014   12   12
2015   12   12
2016   12   12
2017   12   12
2018   12   12
2019   12   12
2020   12   12
2021   12   12
2022   12   12
2023   12   12
2024   12   12
2025   12   12
2026    9    9

Incomplete state-years:


,state_clean,year,cpi_annual_average,months_available
16,johor,2026,141.555556,9
33,kedah,2026,130.500000,9
50,kelantan,2026,130.666667,9
67,melaka,2026,132.855556,9
84,negeri sembilan,2026,136.066667,9
101,pahang,2026,135.544444,9
118,perak,2026,130.211111,9
135,perlis,2026,127.444444,9
152,pulau pinang,2026,139.411111,9
169,sabah,2026,125.822222,9



Duplicate state-year records: 0


In [38]:
# Retain complete calendar years only
cpi_complete = cpi_annual[
    cpi_annual["months_available"] == 12
][
    ["state_clean", "year", "cpi_annual_average"]
].copy()

# Merge annual CPI into state master
state_master = state_master.merge(
    cpi_complete,
    on=["state_clean", "year"],
    how="left",
    validate="one_to_one"
)

# Final validation
print("Shape:", state_master.shape)
print("Geographic categories:", state_master["state_clean"].nunique())
print("Years:", state_master["year"].min(), "-", state_master["year"].max())

print("Duplicate state-year records:",
      state_master.duplicated(["state_clean", "year"]).sum())

print("\nIndicator coverage:")
print(state_master.notna().sum())

print("\n2024 sample:")
display(state_master[state_master["year"] == 2024].head())

# Export
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

state_master.to_csv(
    PROCESSED_DIR / "KAWASAN_Unified_State_Dataset_2010_2025.csv",
    index=False
)

state_master.to_excel(
    PROCESSED_DIR / "KAWASAN_Unified_State_Dataset_2010_2025.xlsx",
    index=False
)

print("\nUnified State-Level Dataset exported successfully!")

Shape: (256, 12)
Geographic categories: 16
Years: 2010 - 2025
Duplicate state-year records: 0

Indicator coverage:
state_clean                      256
year                             256
hh_internet_access               112
icths_internet                   128
water_access                     222
water_consumption_domestic       210
water_consumption_nondomestic    210
wages_median                     256
wages_mean                       256
gdp_total                        168
labour_productivity              176
cpi_annual_average               256
dtype: int64

2024 sample:


,state_clean,year,hh_internet_access,icths_internet,water_access,water_consumption_domestic,water_consumption_nondomestic,wages_median,wages_mean,gdp_total,labour_productivity,cpi_annual_average
14,johor,2024,99.0,98.9,99.9,901.0,610.0,2582,3414,158348.264,81214.0,136.133333
30,kedah,2024,96.4,97.4,98.3,499.0,283.0,2051,3053,54101.506,83295.0,127.200000
46,kelantan,2024,94.9,97.3,73.5,164.0,80.0,1914,3091,28599.018,71838.0,129.291667
62,melaka,2024,96.8,98.5,100.0,242.0,215.0,2528,3469,48844.201,110840.0,128.858333
78,negeri sembilan,2024,96.4,98.6,99.9,323.0,237.0,2503,3626,54335.264,108180.0,131.066667



Unified State-Level Dataset exported successfully!


In [39]:
import shutil
from pathlib import Path

ANALYTICS_DIR = PROJECT_DIR / "data" / "analytics_dataset"
ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)

datasets = [
    "KAWASAN_Unified_District_Dataset_2024",
    "KAWASAN_Unified_State_Dataset_2010_2025",
    "KAWASAN_GDP_District_2015_2020"
]

for dataset in datasets:
    for extension in [".csv", ".xlsx"]:
        source = PROCESSED_DIR / f"{dataset}{extension}"
        destination = ANALYTICS_DIR / f"{dataset}{extension}"

        if source.exists():
            shutil.copy2(source, destination)
            print(f"Exported: {destination.name}")
        else:
            print(f"Missing: {source.name}")

print("\nAnalytics dataset folder:", ANALYTICS_DIR)
print("Total files:", len(list(ANALYTICS_DIR.iterdir())))

Exported: KAWASAN_Unified_District_Dataset_2024.csv
Exported: KAWASAN_Unified_District_Dataset_2024.xlsx
Exported: KAWASAN_Unified_State_Dataset_2010_2025.csv
Exported: KAWASAN_Unified_State_Dataset_2010_2025.xlsx
Exported: KAWASAN_GDP_District_2015_2020.csv
Exported: KAWASAN_GDP_District_2015_2020.xlsx

Analytics dataset folder: c:\Users\Muhd Haziq\Desktop\KAWASAN\data\analytics_dataset
Total files: 6
